Problem 1:
Implement a feed-forward neural network in PyTorch.
1. Construct a synthetic regression dataset with a train and test dataset, where a
one-layer network does not show good performance on the test dataset, while a
network with more layers shows good performance.
2. Can you propose a regularization term that reduces the performance gap between
the test and train datasets for the larger network? Show empirically that your
regularizer works.

In [81]:
import torch
import torch.nn as nn

class SimpleNN(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, output_size)

    def forward(self, x):
        out = self.fc1(x)
        return out

class MultiLayerNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiLayerNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out = torch.relu(self.fc1(x))
        out = self.fc2(out)
        return out

In [82]:
# Construct synthetic regression dataset
class SyntheticRegressionDataset(torch.utils.data.Dataset):
    def __init__(self, num_samples, input_size):
        self.X = torch.randn(num_samples, input_size)
        # self.y = torch.sum(self.X, dim=1) * 0.4 + torch.randn(num_samples) * 0.1 # Add some noise
        self.y = torch.sin(torch.sum(self.X, dim=1)) + torch.randn(num_samples) * 0.1 # Add some noise

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [83]:
input_size = 10
output_size = 1

dataset = SyntheticRegressionDataset(1000, input_size)
# test_dataset = SyntheticRegressionDataset(200, input_size)
# split dataset into train and test
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

one_layer_model = SimpleNN(input_size, output_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(one_layer_model.parameters(), lr=0.01)

In [84]:
# Train the one-layer model
for epoch in range(100):
    train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = one_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0) 
    print(f'Epoch [{epoch+1}/100], Loss: {train_loss / len(train_loader):.4f}')
# Evaluate the one-layer model
one_layer_model.eval()
with torch.no_grad():
    test_loss = 0
    for X_batch, y_batch in test_loader:
        outputs = one_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        test_loss += loss.item() * X_batch.size(0)
print(f'One-layer model test loss: {test_loss / len(test_loader):.4f}')
one_layer_train_loss = train_loss / len(train_loader)
one_layer_test_loss = test_loss / len(test_loader)

Epoch [1/100], Loss: 17.9429
Epoch [2/100], Loss: 16.2431
Epoch [3/100], Loss: 16.2405
Epoch [4/100], Loss: 16.1642
Epoch [5/100], Loss: 16.2227
Epoch [6/100], Loss: 16.1896
Epoch [7/100], Loss: 16.2271
Epoch [8/100], Loss: 16.2240
Epoch [9/100], Loss: 16.1990
Epoch [10/100], Loss: 16.2668
Epoch [11/100], Loss: 16.2021
Epoch [12/100], Loss: 16.2446
Epoch [13/100], Loss: 16.2291
Epoch [14/100], Loss: 16.2369
Epoch [15/100], Loss: 16.2975
Epoch [16/100], Loss: 16.2476
Epoch [17/100], Loss: 16.2972
Epoch [18/100], Loss: 16.2506
Epoch [19/100], Loss: 16.2534
Epoch [20/100], Loss: 16.3918
Epoch [21/100], Loss: 16.3103
Epoch [22/100], Loss: 16.2215
Epoch [23/100], Loss: 16.3030
Epoch [24/100], Loss: 16.2761
Epoch [25/100], Loss: 16.2036
Epoch [26/100], Loss: 16.1770
Epoch [27/100], Loss: 16.2607
Epoch [28/100], Loss: 16.1747
Epoch [29/100], Loss: 16.2828
Epoch [30/100], Loss: 16.1750
Epoch [31/100], Loss: 16.1668
Epoch [32/100], Loss: 16.2092
Epoch [33/100], Loss: 16.2897
Epoch [34/100], Los

In [85]:
# Train multi-layer model
hidden_size = 20
multi_layer_model = MultiLayerNN(input_size, hidden_size, output_size)
optimizer = torch.optim.Adam(multi_layer_model.parameters(), lr=0.01)


In [86]:
for epoch in range(100):
    train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = multi_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    print(f'Epoch [{epoch+1}/100], Loss: {train_loss / len(train_loader):.4f}')
# Evaluate the multi-layer model
multi_layer_model.eval()
with torch.no_grad():
    test_loss = 0
    for X_batch, y_batch in test_loader:
        outputs = multi_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        test_loss += loss.item() * X_batch.size(0)
print(f'Multi-layer model test loss: {test_loss / len(test_loader):.4f}')
multi_layer_train_loss = train_loss / len(train_loader)
multi_layer_test_loss = test_loss / len(test_loader)

Epoch [1/100], Loss: 17.0954
Epoch [2/100], Loss: 16.1176
Epoch [3/100], Loss: 15.8846
Epoch [4/100], Loss: 15.7772
Epoch [5/100], Loss: 15.7435
Epoch [6/100], Loss: 15.5244
Epoch [7/100], Loss: 15.4476
Epoch [8/100], Loss: 15.1259
Epoch [9/100], Loss: 15.2652
Epoch [10/100], Loss: 15.2670
Epoch [11/100], Loss: 14.7515
Epoch [12/100], Loss: 14.2390
Epoch [13/100], Loss: 13.8584
Epoch [14/100], Loss: 13.7090
Epoch [15/100], Loss: 13.4698
Epoch [16/100], Loss: 13.2738
Epoch [17/100], Loss: 13.1935
Epoch [18/100], Loss: 12.8297
Epoch [19/100], Loss: 12.4954
Epoch [20/100], Loss: 12.1352
Epoch [21/100], Loss: 11.7330
Epoch [22/100], Loss: 11.4522
Epoch [23/100], Loss: 10.3893
Epoch [24/100], Loss: 10.1020
Epoch [25/100], Loss: 9.8476
Epoch [26/100], Loss: 9.4023
Epoch [27/100], Loss: 8.7519
Epoch [28/100], Loss: 8.8107
Epoch [29/100], Loss: 8.5642
Epoch [30/100], Loss: 8.3947
Epoch [31/100], Loss: 8.4809
Epoch [32/100], Loss: 8.7028
Epoch [33/100], Loss: 8.8512
Epoch [34/100], Loss: 8.4707

In [87]:
print(f'One-layer model train loss: {one_layer_train_loss:.4f}, test loss: {one_layer_test_loss:.4f}')
print(f'Multi-layer model train loss: {multi_layer_train_loss:.4f}, test loss: {multi_layer_test_loss:.4f}')

One-layer model train loss: 16.1780, test loss: 14.9698
Multi-layer model train loss: 2.3565, test loss: 1.7508


Problem 2:
Derive a hypothesis class H and a data distribution P such that there exists a hypothesis
h ∈ H for which the training error is zero, but the expected test error is no better than
random guessing.
1. Define a finite hypothesis class H with at least two hypotheses and describe a data
distribution P on (X , Y).
2. Construct a small training set drawn from P where one hypothesis hoverfit fits the
training data perfectly, achieving zero training error.
3. Explain why, under the chosen P, this hypothesis does no better than random
guessing on unseen test data.

Let $P$ be a distribution on $(X, Y)$,  
where $X \in \mathbb{R}^2$ and $Y \in \{0, 1\}$ is a binary label.

$X = (X_1, X_2)$ there exists a boundary in $X_{1,boundary}$ and $X_{2,boundary}$ such that:
- If $X_1 \ge X_{1,boundary}$ and $X_2 \ge X_{2,boundary}$, then $Y = 0$.
- If $X_1 \le X_{1,boundary}$ and $X_2 \le X_{2,boundary}$, then $Y = 0$.
- For all other combinations, $Y = 1$.

Then, 
1. Hypothesis class $H$ may contain:
- $h_1(X) = 0$ if $X_1 \ge X_{1,boundary}$ else $1$
- $h_2(X) = 0$ if $X_1 \le X_{1,boundary}$ 

2. Let the training set $S_N$ be defined as:  
 $S_{N, small region} = \{((X_1, X_2), Y) |  X_2 < X_{2,boundary}\}$

then, $h_2$ fits perfectly.

3. However, if a better training set is constructed as:  
$S_{N, large region} = \{((X_1, X_2), Y) |  X_{1,boundary} - \delta < X_1 < X_{1,boundary} + \delta, X_{2,boundary} - \delta < X_2 < X_{2,boundary} + \delta\}$


$h_2$ or $h_1$ will perform no better than random.

